# Solver comparison — calibrate ONE sample under 3 solvers (array-stack driver)

<details>
<summary>A fork of [`Convergence_Run.ipynb`](Convergence_Run.ipynb). Instead of sweeping a</summary>

population of random parameter sets, this draws **one** set of initial conditions (the
first Latin-Hypercube sample) and runs the *same staged calibration* under each of three
integrators — **euler**, **rk4**, **dopri5** — toward the fixed physiological **twin**
target state.

Each "run" in the population artifact is a solver (not a population member). Pruned to
exactly what the manuscript shows for Reviewer 1 §2.4 (R1-C11): the per-solver residual /
calibrated-parameter agreement / wall-clock breakout, the LaTeX comparison table, and the
comparison figure. The generic population views (scope report, error summary, boxplot,
convergence-trace plots, timing table) were removed — see `run_convergence/serial.ipynb`
for those.

Solver type is honoured only on the step-independent (SI) stack via `runner.run`
(`simulationParams['solver']`); the legacy stack is `diffrax.Euler` only. Config lives in
`config/scenarios/sepsis_linear.json` (linear controller law, 370 internal runs per
calibration) paired with `config/models/cvModel_linear.json`. The truncated
`solver_compare.json` clone is NOT usable here — its 13-run stage stack leaves a ~70%
residual, so it cannot support the integrator-equivalence claim.

</details>

In [ ]:
# region -> runConfig — the single run-configuration surface (device/precision applied before JAX)
####################################################################################################
# runConfig — the ONE place run configuration lives (project rule: see repo-root CLAUDE.md).
# Defined first so the device/precision block can be applied before JAX initialises below.
####################################################################################################
runConfig = {
    # --- file references ---------------------------------------------------
    "model":    "cvModel_linear.json",  # config/models/ cardiopulmonary model (16 calibration controllers),
                                        #   linear-law controllers to match the scenario below.
    "scenario": "sepsis_linear.json",   # config/scenarios/ full linear-law stage stack (370 internal runs).
                                        #   Was solver_compare.json -- a 13-run truncated clone that does
                                        #   NOT converge (~70% residual), so it cannot support the
                                        #   integrator-equivalence claim.
    "mode":     "calibration",          # each solver runs the SAME staged calibration

    # --- pipeline phases (run + plot separable; see runConfig.run / runConfig.plot) ------
    "run":  False,   # Phase 1 — execute the solver sweep + save the population artifact
    "plot": True,   # Phase 2 — load from the saved file + analyse + plot

    # --- device / precision (applied in the Imports cell, before `import jax`) ---------
    "device": {
        "useGpu":    False,       # True -> CUDA device (requires `jax[cuda12]` in the venv)
        "precision": "float64",   # run precision: "float64" (parity/reference) or "float32"
    },

    # --- integration stack + solver ---------------------------------------
    # "SI" -> the step-independent stack, so runner.run honours the solver (the legacy stack
    # is diffrax.Euler only). `solver` is the fallback for buildSimulationParams; `solvers`
    # below is the actual set compared (each rebuilds simulationParams with its own solver).
    "stack":     "SI",
    "solver":    {"type": "euler"},
    "solvers": {
        "euler":  {"type": "euler"},
        "rk4":    {"type": "rk4"},
        # dopri5 grinds tiny steps across the valve discontinuities (~260/beat), so a 10 s
        # solve needs a few thousand steps; 8192 was marginally too low (diverged), but a huge
        # ceiling (1e6) blew up RAM via diffrax's per-solve buffers. 8192 (scenario default) is enough and lightest.
        "dopri5": {"type": "dopri5", "rtol": 1e-6, "atol": 1e-9, "maxSteps": 8192},
    },

    # --- calibration overrides ---------------------------------------------
    # Per-key overrides of the scenario's calibration section (dict values merge one
    # level deep), e.g. {"maxCubicFactor": 5.0} or {"adaptive": {"maxLoops": 12}}.
    # {} = use the scenario as-is.
    "calibration": {
    },

    # --- sample selection --------------------------------------------------
    # The comparison calibrates a single set of initial conditions: LHS row `sampleIndex`
    # (seed fixes the draw). nrModels only sizes the draw so that row exists.
    "population": {
        "nrModels":    1,         # LHS rows drawn (>= sampleIndex + 1)
        "seed":        0,         # LHS reproducibility
        "sampleIndex": 0,         # which drawn row to calibrate under every solver
        "errorTarget": 0.5,       # SUCCESS if max |relative error| (%) <= this
    },

    # --- analysis ----------------------------------------------------------
    "analysis": {
        "atm": 760.0,             # atmospheric offset baked into absolute-pressure signals (gauge = raw - atm)
        # Paper-figure typography, same convention as run_test/calibration_compare.ipynb.
        # figSize is chosen so `fontSize` lands at ~9pt once the PNG is scaled to the
        # manuscript's 8.5 cm column: 18 pt x (3.35 in / 6.7 in) = 9 pt.
        "fontSize": 18,
        "figSize":  [6.7, 4.6],
    },

    # --- output ------------------------------------------------------------
    "output": {"save": True, "path": "notebookData/test", "name": "results_solver_compare.h5",
               "logProgress": True},   # persist the live-progress trace into the .h5

    # --- paper artifacts (R1-C11 integrator comparison) --------------------
    # Phase-2 emits the per-solver comparison table + figure straight into the manuscript
    # revision tree when `emit` is on (single config surface: paths are runConfig keys,
    # consumed with .get(key, default) fallbacks, never literals in the analysis cells).
    "paper": {
        "emit":         True,
        "generatedDir": "EFC_Paper/revision/generated",
        "imagesDir":    "EFC_Paper/revision/Images",
        "tableName":    "integratorComparison.tex",
        "figName":      "integratorComparison.png",
    },

    # post-processing / per-run HDF5 sim-artifact are off here: the population
    # file is the artifact, and errors read straight from the raw observation signals.
    "postProcessing": None,
    "requested":      None,
    "plots":          [],

    # --- printing ------------------------------------------------------------
    "printStatus":    True,       # per-run integrator timing lines (runSimulationArray)
    "printEveryPct":  10,         # unused here (one run per solver); kept for parity
    "progressEvery":  1000,          # within-run sim-time convergence line every N sim-seconds (0 = off;
                                  #   the live view here is the per-solver convergence line in the run loop)

    # --- integration numerics (override scenario shared.integration; single config surface) ---
    "runTime": 10,       # simulated seconds per internal run
    "dt":      0.0005,   # integrator step
    "dtDense": 0.01,     # dense-output sampling step
}
# endregion

## Imports

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

# ---- device / precision (from runConfig, MUST run before JAX initialises) -------
useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

import os
if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    # GPU memory hygiene (must precede `import jax`): grow on demand instead of grabbing
    # ~75% of VRAM up front (so JAX coexists with the display on a small shared card), and
    # hand freed buffers back to the driver so the cleanup cell / del actually releases VRAM.
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", precision == "float64")

import library.run.runner as runner
import library.run.progress as progressLib   # standard live-progress line + record log            # run orchestration (modes + stage stacks)
import library.utils as utils
from library.hdf5 import schema_pop                        # population artifact (init_population / add_run)
import numpy as np
from scipy.stats import qmc                        # Latin Hypercube sampling (SALib not required)
import pandas as pd
import matplotlib.pyplot as plt
import time
import traceback

np.set_printoptions(suppress=True)
print("devices:", jax.devices(), "| x64:", jax.config.jax_enable_x64)
# endregion

## Assemble `simulationParams` + load the convergence config

<details>
<summary>`runner.buildSimulationParams` expands the slim `runConfig` + scenario into the legacy</summary>

`simulationParams` shape. The `convergence` block of the scenario carries the parameter
bounds (LHS) and the observation list (the controllers' `varTarget`s + `V_Vs`).

</details>

In [ ]:
# region -> assemble simulationParams + load the convergence config (params, bounds, observations)
scenario = utils.loadScenario(runConfig["scenario"])
simulationParams = runner.buildSimulationParams(runConfig, scenario)

conv          = scenario["convergence"]
twin          = scenario["shared"]["twin"]["twinTargets"]
volDist       = scenario["shared"]["twin"]["volumeDistribution"]
observations  = conv["observations"]
pop           = runConfig["population"]
outPath       = os.path.join(runConfig["output"]["path"], runConfig["output"]["name"])

# LHS parameter bounds. Scenarios that define their own parameter sweep carry them under
# `convergence.parameters`; the sepsis-family scenarios no longer do (their population block
# samples clinical targets per phenotype instead). In that case fall back to the `problem`
# record the population file itself stores -- schema_pop.init_population persisted the exact
# names + bounds the artifact was swept with, so Phase 2 reproduces from the file alone.
if "parameters" in conv:
    paramBounds = conv["parameters"]
    boundsFrom  = f"scenario {runConfig['scenario']} (convergence.parameters)"
elif os.path.exists(outPath):
    problem     = schema_pop.read_population(outPath).problem
    paramBounds = dict(zip(problem["names"], problem["bounds"]))
    boundsFrom  = f"artifact {outPath} (problem attr)"
else:
    raise KeyError(
        f"no LHS parameter bounds: scenario '{runConfig['scenario']}' has no "
        f"convergence.parameters and no artifact exists at {outPath} to recover them from. "
        f"Add a convergence.parameters block to the scenario to run Phase 1.")

param_names = list(paramBounds.keys())
bounds      = np.array([paramBounds[p] for p in param_names], dtype=float)  # (P, 2)

print(f"nrModels = {pop['nrModels']} | params = {len(param_names)} | observations = {len(observations)}")
print(f"bounds <- {boundsFrom}")
print(f"output -> {outPath}")
# endregion

## Twin-target array + atmospheric offsets

<details>
<summary>Each observation has a target derived from the twin state: pressures from the twin pressure</summary>

targets, stroke volume from `CO/HR`, compartment volumes from `volumeDistribution ×
TotalBloodVolume`, and `Cyc_HC` from `60/HR`. Absolute-pressure observations carry the model's
+760 mmHg atmospheric offset, which we subtract to compare in gauge mmHg (amplitudes, volumes
and SV carry no offset).

</details>

In [ ]:
# region -> twin-target array + atmospheric offsets per observation
ATM = runConfig["analysis"]["atm"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

def obsTarget(name):
    """Twin target for an observation (gauge units), or None if untargeted.

    Matches Convergence_Population.ipynb's targetsArray: the capillary means are
    derived as a pressure DROP from the upstream arterial target, not as absolute
    means -- avg_P_Cs = (Sys_P_As - amp_P_As) - twin.avg_P_Cs (systemic diastolic
    minus the configured drop), avg_P_Cp = Dia_P_Ap - twin.avg_P_Cp.
    """
    TBV = twin["TotalBloodVolume"]
    direct = {
        "avg_P_Vs": twin["CVP"],
        "avg_P_Cs": twin["Sys_P_As"] - twin["amp_P_As"] - twin["avg_P_Cs"],
        "avg_P_Cp": twin["Dia_P_Ap"] - twin["avg_P_Cp"],
        "keep_max_P_As": twin["Sys_P_As"], "keep_max_P_Ap": twin["Sys_P_Ap"],
        "keep_min_P_Ap": twin["Dia_P_Ap"], "amp_P_As": twin["amp_P_As"],
        "keep_SV_Hl": twin["CO"] / twin["HR"], "Cyc_HC": 60.0 / twin["HR"],
        "V_Vs": volDist["Vs"] * TBV,
    }
    if name in direct:
        return direct[name]
    if name.startswith("avg_V_"):
        return volDist[name[len("avg_V_"):]] * TBV
    return None

targetArr = np.array([obsTarget(o) if obsTarget(o) is not None else np.nan for o in observations])
offsetArr = np.array([obsOffset(o) for o in observations])
pd.DataFrame({"observation": observations, "target": targetArr, "offset": offsetArr})
# endregion

## Latin Hypercube sample of the parameter space

<details>
<summary>`sampled_params` is an `(nrModels, P)` matrix scaled into each parameter's `[min, max]`</summary>

bounds. Each row becomes the initial value of the corresponding controlled state for one
population member (`runner.run(..., stateOverrides=row)`).

</details>

In [ ]:
# region -> Latin Hypercube sample of the parameter space
if runConfig["run"]:
    sampler = qmc.LatinHypercube(d=len(param_names), seed=pop["seed"])
    unit = sampler.random(n=pop["nrModels"])                      # (N, P) in [0, 1)
    sampled_params = qmc.scale(unit, bounds[:, 0], bounds[:, 1])   # (N, P) in [min, max]
    print("sampled_params shape:", sampled_params.shape)
    pd.DataFrame(sampled_params, columns=param_names).head()
# endregion

## Run the population

<details>
<summary>For each sample: inject the draw as `stateOverrides`, run the staged calibration, read each</summary>

observation's steady-state value (last completed-cycle value) from `results`, and append the
run to the population file. `schema_pop.add_run` sanitizes any NaN / diverging run to
`BAD_RUN_SENTINEL`; solver exceptions are caught and recorded as a sentinel run so the sweep
never aborts. The population file is initialized lazily from the first run so its `state_names`
and `model_structure` exactly match the runtime.

</details>

In [ ]:
# region -> calibrate one LHS sample under each solver, save each as a run in the population artifact
if runConfig["run"]:
    def steadyState(results, name):
        """Gauge steady-state value of an observation (last value minus atmospheric offset)."""
        return utils.steadyState(results, name, ATM)

    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)

    # ONE set of initial conditions (LHS row `sampleIndex`) calibrated under every solver.
    # Each "run" in the population file is a solver: run_id = index i, labels in solverLabels.
    solverLabels = list(runConfig["solvers"].keys())
    sampleIndex  = runConfig["population"].get("sampleIndex", 0)
    override0    = {p: float(sampled_params[sampleIndex, j]) for j, p in enumerate(param_names)}  # shared init
    nRuns        = len(solverLabels)

    # Bound up-front: a solver that raises before returning would otherwise leave
    # `modelStructure` undefined and the init_population call below would mask the real
    # solver error behind a NameError.
    modelStructure = None
    obsMatrix  = np.full((nRuns, len(observations)), np.nan)   # in-memory mirror for plots
    rawSignals = list(dict.fromkeys(list(observations) + list(param_names)))  # obs + swept params -> raw
    runWall    = np.full(nRuns, np.nan)                        # per-solver wall clock (s), persisted
    initialized = False
    # Standard live-progress line: each solver's achieved obs-space |rel err| vs twin targets,
    # so the solvers' accuracies are directly comparable (and logged for later comparison).
    reporter = progressLib.ProgressReporter(logEnabled=runConfig["output"].get("logProgress", True))
    t0 = time.time()

    for i, label in enumerate(solverLabels):
        # rebuild simulationParams with this solver (only `solver` differs; init/targets identical)
        solverParams = runner.buildSimulationParams(
            {**runConfig, "solver": runConfig["solvers"][label]}, scenario)
        tRun = time.time()
        try:
            states, modelObjects, modelStructure, results, structures = runner.run(
                solverParams, stateOverrides=override0)
            rawTraces = {n: np.asarray(results[n]) for n in rawSignals if n in results}
            obsMatrix[i] = [steadyState(results, o) if o in results else np.nan for o in observations]
            finalState = states
        except Exception as e:
            traceback.print_exc()
            print(f"  [{label}] solver exception: {e}")
            rawTraces = {n: np.array([schema_pop.BAD_RUN_SENTINEL]) for n in rawSignals}
            finalState = {p: schema_pop.BAD_RUN_SENTINEL for p in param_names}
        runWall[i] = time.time() - tRun

        # The population file can only be initialised from a successful run (it needs the
        # runtime model structure). If the FIRST solver fails there is nothing to key the
        # artifact on, so fail loudly with the solver's own error rather than silently.
        if runConfig["output"]["save"] and not initialized and modelStructure is None:
            raise RuntimeError(
                f"[{label}] failed before producing a model structure -- cannot initialise "
                f"{outPath}. Fix the solver error above (traceback printed) and re-run.")

        if runConfig["output"]["save"]:
            if not initialized:
                schema_pop.init_population(
                    outPath,
                    param_names=param_names,
                    state_names=list(finalState.keys()),
                    observation_names=observations,
                    sampled_params=sampled_params[sampleIndex:sampleIndex + 1],
                    model_structure=utils.modelStructureJSON(modelStructure),
                    problem={"names": param_names, "bounds": bounds.tolist(),
                             "num_vars": len(param_names)},
                    conf=runConfig, meta={"twinTargets": twin, "solverLabels": solverLabels,
                                          "sampleIndex": sampleIndex})
                initialized = True
            schema_pop.add_run(outPath, run_id=str(i), raw=rawTraces, final_state=finalState)

        print(f"{np.round(time.time() - tRun, 4)}s -> [{i}] {label} "
              f"({runConfig['solvers'][label]['type']}) calibrated ({time.time() - t0:.0f}s total)")
        reporter.emit(kind="step", label=f"[{label}] solver {i + 1}/{nRuns}",
                      done=i + 1, total=nRuns, elapsedWall=time.time() - t0,
                      stats=progressLib.relErrorStats(obsMatrix[i:i + 1], targetArr))

    if runConfig["output"]["save"]:
        schema_pop.write_timings(outPath, runWall, meta={
            "device":       "gpu" if useGpu else "cpu",
            "precision":    precision,
            "solver":       "|".join(solverLabels),
            "solverLabels": solverLabels,
            "dt":           simulationParams["dt"],
            "runTime":      simulationParams["runTime"],
            "nrModels":     nRuns,
            "stack":        runConfig["stack"],
            "total_wall":   time.time() - t0,
        })
        # persist the live-progress trace (per-solver convergence lines) for comparison
        schema_pop.write_progress(outPath, reporter.records, meta={
            "model": runConfig["model"], "scenario": runConfig["scenario"],
            "mode": runConfig["mode"], "solver": "|".join(solverLabels),
            "nrModels": nRuns, "stack": runConfig["stack"]})

    print(f"Solver comparison complete in {time.time() - t0:.0f}s -> {outPath}")
# endregion

# Phase 2 — Load & analyse (from the saved file)

<details>
<summary>Everything below reconstructs from the population `.h5` — no in-session run state</summary>

is required. After a kernel restart you can run the setup cells (config →
`simulationParams` → targets → LHS, all scenario-only and cheap) then the **load**
cell below and the comparison cells, without re-running the sweep.

The load cell rebuilds `obsMatrix` (each observation's steady-state value = last saved
raw value minus its atmospheric offset), `paramMatrix` (each solver's run-end calibrated
parameters) and the per-run timings (`runWall`, `timingMeta`) written in Phase 1.

</details>

In [ ]:
# region -> load everything the analysis needs straight from the saved population file
if runConfig["plot"]:
    # --- load everything the analysis below needs, straight from the saved file ----------
    # Reconstructs the in-memory run state (obsMatrix, paramMatrix) + per-run timings so
    # Phase 2 is independent of Phase 1 having run in this kernel.
    import h5py

    with h5py.File(outPath, "r") as f:
        runIds = list(f["run_ids"].asstr()[:])
        obsMatrix = np.full((len(runIds), len(observations)), np.nan)
        paramMatrix = np.full((len(runIds), len(param_names)), np.nan)   # run-end swept/calibrated params
        for rid in runIds:
            i = int(rid)
            grp = f[f"runs/{rid}/raw"]
            for j, o in enumerate(observations):
                if o in grp:
                    obsMatrix[i, j] = float(np.asarray(grp[o]).flat[-1]) - obsOffset(o)
            for j, p in enumerate(param_names):        # a run whose parameter leaves scope is BAD too
                if p in grp:
                    paramMatrix[i, j] = float(np.asarray(grp[p]).flat[-1])
    runWall, timingMeta = schema_pop.read_timings(outPath)
    print(f"loaded {obsMatrix.shape[0]} runs from {outPath} | timings: {np.isfinite(runWall).sum()} timed")
# endregion

# Phase 2 — Integrator comparison (Reviewer 1, §2.4 / R1-C11)

<details>
<summary>The Reviewer 1 §2.4 breakout: from the reloaded population file, per solver (euler / rk4 /</summary>

dopri5) compute the calibration residual (max and mean $|$relative error$|$ % across the
targeted observations), the calibrated-parameter agreement (per-parameter max relative spread
across solvers, plus each solver's run-end deviation from the Euler reference), and the
wall-clock cost with its slowdown relative to Euler. Steps-per-beat is deliberately omitted —
the step-grinding is described in prose only.

</details>

In [ ]:
# region -> R1-C11: per-solver residual / calibrated-param agreement / wall-clock breakout
if runConfig["plot"]:
    # run order == solver order (population run_id i is solvers[i]); single config surface.
    solverLabels = list(runConfig["solvers"].keys())          # euler, rk4, dopri5
    solverTypes  = {l: runConfig["solvers"][l]["type"] for l in solverLabels}

    targetedMask = ~np.isnan(targetArr)
    tgtT         = targetArr[targetedMask]
    obsCompare   = [o for o, keep in zip(observations, targetedMask) if keep]

    # --- residual: per-solver signed relative error (%) across the targeted observations ---
    relErrPerSolver = {}
    for i, label in enumerate(solverLabels):
        vals = obsMatrix[i, targetedMask]
        relErrPerSolver[label] = (vals - tgtT) / tgtT * 100.0

    # --- calibrated-parameter agreement across the three solvers -------------------------
    pv = paramMatrix                                          # (nSolvers, nParams) run-end params
    with np.errstate(divide="ignore", invalid="ignore"):
        # per-param max relative spread across solvers = (max-min)/|mean| * 100
        spreadPerParam = (np.nanmax(pv, axis=0) - np.nanmin(pv, axis=0)) \
                         / np.abs(np.nanmean(pv, axis=0)) * 100.0
        # per-solver run-end deviation of each calibrated param from the Euler reference (%)
        ref = pv[0]                                           # Euler = solverLabels[0]
        paramDev = np.abs(pv - ref) / np.abs(ref) * 100.0     # (nSolvers, nParams)
    maxSpread, meanSpread = float(np.nanmax(spreadPerParam)), float(np.nanmean(spreadPerParam))

    # --- wall-clock (one staged calibration per solver -> per-solver total == s/run) ------
    solverWall = {label: float(runWall[i]) for i, label in enumerate(solverLabels)}
    base       = solverWall[solverLabels[0]]
    slowdown   = {label: solverWall[label] / base for label in solverLabels}

    solverStats = pd.DataFrame([{
        "solver":            label,
        "type":              solverTypes[label],
        "max_rel_err_%":     float(np.nanmax(np.abs(relErrPerSolver[label]))),
        "mean_rel_err_%":    float(np.nanmean(np.abs(relErrPerSolver[label]))),
        "max_param_dev_%":   float(np.nanmax(paramDev[i])),
        "mean_param_dev_%":  float(np.nanmean(paramDev[i])),
        "s_per_run":         solverWall[label],
        "total_wall_s":      solverWall[label],
        "slowdown_x":        slowdown[label],
    } for i, label in enumerate(solverLabels)]).set_index("solver")

    # --- per-parameter agreement: each solver's run-end value, the cross-solver spread, and
    # --- the deviation from the Euler reference. The table's `Max $\\Delta$p vs Euler` row is
    # --- the column-max of the dev_* columns below.
    paramAgreement = pd.DataFrame({l: pv[i] for i, l in enumerate(solverLabels)}, index=param_names)
    paramAgreement["spread_%"] = spreadPerParam
    for i, l in enumerate(solverLabels):
        if i:                                       # euler is the reference -> dev == 0
            paramAgreement[f"dev_{l}_%"] = paramDev[i]
    paramAgreement.index.name = "param"
    display(paramAgreement)

    print(f"cross-solver calibrated-param spread: max {maxSpread:.3g}% | mean {meanSpread:.3g}%")
    print("slowdown vs " + solverLabels[0] + ": "
          + " : ".join(f"{l} {slowdown[l]:.1f}x" for l in solverLabels))
    solverStats
# endregion

## Integrator comparison — LaTeX table (R1-C11)

<details>
<summary>Renders the per-solver comparison (residual, calibrated-parameter agreement vs the Euler</summary>

reference, and wall-clock slowdown) as a LaTeX `tabular` via `utils.generateLatexTableInline`
— the non-float (`\captionof`) emitter, because the manuscript body is a `multicols`
environment and LaTeX silently drops a real `table` float inside it. The layout is
**transposed** (solvers as columns, metrics as rows) so it fits one 8.5 cm column.
Written to `paper.generatedDir/paper.tableName` (created if absent) when `paper.emit` is on.

</details>

In [ ]:
# region -> R1-C11: LaTeX integrator-comparison table -> paper generatedDir (guarded by paper.emit)
if runConfig["plot"]:
    paperT = runConfig.get("paper", {})
    worstErr = float(np.nanmax([solverStats.loc[l, "max_rel_err_%"] for l in solverLabels]))
    dopSlow  = slowdown[solverLabels[-1]]

    # Cost per unit of INTEGRATED MODEL TIME — the hardware-independent unit used to price the
    # independent-optimiser baselines (each of their forward solves is one runTime-long solve).
    # A staged calibration integrates runner.runCalibrationSI's `simTotal` seconds:
    #   sum(runsToIgnore + runsToSave) over the stages x runTime.
    runTimeS  = simulationParams["runTime"]
    stagesCfg = scenario["calibration"]["stages"]
    nInner    = sum(s["runsToIgnore"] + s["runsToSave"] for s in stagesCfg)   # internal runs
    simTotalS = nInner * runTimeS                                            # simulated s / calibration

    # TRANSPOSED layout: the three solvers are the columns and the metrics are the rows, so
    # the table fits one 8.5 cm column of the paper's two-column body (a 7-column solver-per-row
    # table overflows it). Emitted as a NON-float brace group via generateLatexTableInline —
    # LaTeX drops a real `table` float inside the manuscript's `multicols` environment.
    # No `Type` row: the solver labels ARE the types here, so it just repeats the header.
    metricRows = {
        "Max $|\\varepsilon_r|$ \\%":  [f"{solverStats.loc[l, 'max_rel_err_%']:.3g}"   for l in solverLabels],
        "Mean $|\\varepsilon_r|$ \\%": [f"{solverStats.loc[l, 'mean_rel_err_%']:.3g}"  for l in solverLabels],
        "Max $\\Delta$p vs Euler \\%": [f"{solverStats.loc[l, 'max_param_dev_%']:.3g}" for l in solverLabels],
        f"s / {runTimeS:g}\\,s solve":  [f"{solverStats.loc[l, 's_per_run'] * runTimeS / simTotalS:.3g}"
                                        for l in solverLabels],
        "s / calibration":            [f"{solverStats.loc[l, 's_per_run']:.1f}"       for l in solverLabels],
        "Slowdown $\\times$":          [f"{slowdown[l]:.1f}"                           for l in solverLabels],
    }

    # Caption kept SHORT: the mechanism (why Dopri5 grinds, why the per-solve unit matters) is
    # already in the surrounding appendix prose, so it does not belong in the caption too.
    integratorTable = utils.generateLatexTableInline(
        metricRows,
        [""] + [f"{l} ({solverTypes[l]})" if solverTypes[l] != l else l for l in solverLabels],
        "table:integratorComparison",
        f"Integrator comparison for one staged calibration driven to the same twin targets. "
        f"$|\\varepsilon_r|$ is the relative error against the targets, $\\Delta$p the run-end "
        f"deviation of each calibrated parameter from the Euler reference. One calibration "
        f"integrates {simTotalS:g}\\,s of simulated time ({nInner} runs of {runTimeS:g}\\,s), so "
        f"\\emph{{s / {runTimeS:g}\\,s solve}} is the cost of a single forward solve. All three "
        f"agree to within {worstErr:.2g}\\%, but Dopri5 is $\\sim${dopSlow:.0f}$\\times$ slower.")

    if paperT.get("emit", False):
        genDir = paperT.get("generatedDir", "EFC_Paper/revision/generated")
        os.makedirs(genDir, exist_ok=True)
        outTable = os.path.join(genDir, paperT.get("tableName", "integratorComparison.tex"))
        with open(outTable, "w") as fh:
            fh.write(integratorTable)
        print(f"LaTeX table -> {outTable}\n")
    print(integratorTable)
# endregion

## Integrator comparison — figure (R1-C11)

<details>
<summary>One panel: the signed relative error of every targeted observation, grouped as three bars</summary>

per observation (euler / rk4 / dopri5), against the $\pm$`errorTarget`% band. The wall-clock
cost is deliberately NOT plotted — those numbers are already the `s/run` and `Slowdown` rows
of the comparison table, and a second panel only shrank the residual panel below legibility
at the manuscript's 8.5 cm column width. Saved to `paper.imagesDir/paper.figName` when
`paper.emit` is on, alongside the inline view.

</details>

In [ ]:
# region -> R1-C11: comparison figure (per-obs residual grouped by solver)
if runConfig["plot"]:
    paperF = runConfig.get("paper", {})
    # Typography from runConfig, same convention as calibration_compare.ipynb: author the
    # figure at `figSize` with `fontSize` type, so after \includegraphics scales it to the
    # manuscript's 8.5 cm column the labels land near the 10 pt body size.
    fs      = runConfig["analysis"].get("fontSize", 18)
    figSize = tuple(runConfig["analysis"].get("figSize", [6.7, 4.6]))
    # categorical, reproducible solver colours (tab palette; not a config value)
    solverColors = [plt.get_cmap("tab10")(k) for k in range(len(solverLabels))]

    # Single panel only: the wall-clock/slowdown numbers live in the comparison TABLE, so a
    # second cost panel just duplicated them and shrank this one below legibility.
    fig, axE = plt.subplots(figsize=figSize)

    # --- signed relative error per targeted observation, grouped by solver ----------------
    x = np.arange(len(obsCompare))
    barW = 0.8 / len(solverLabels)
    for k, label in enumerate(solverLabels):
        axE.bar(x + (k - (len(solverLabels) - 1) / 2) * barW,
                relErrPerSolver[label], barW, color=solverColors[k],
                label=f"{label} ({solverTypes[label]})")
    axE.axhline(0.0, color="k", lw=1.0)
    axE.axhline(pop["errorTarget"], color="r", ls="--", lw=1.0,
                label=f"$\\pm${pop['errorTarget']}% target")
    axE.axhline(-pop["errorTarget"], color="r", ls="--", lw=1.0)
    axE.set_xticks(x)
    axE.set_xticklabels(utils.labelsFor(obsCompare, "latex"), rotation=90, fontsize=fs)
    axE.tick_params(axis="y", labelsize=fs)
    axE.set_ylabel("relative error (%)", fontsize=fs)
    axE.legend(fontsize=fs - 4, ncol=2)
    plt.tight_layout()

    if paperF.get("emit", False):
        imgDir = paperF.get("imagesDir", "EFC_Paper/revision/Images")
        os.makedirs(imgDir, exist_ok=True)
        outFig = os.path.join(imgDir, paperF.get("figName", "integratorComparison.png"))
        plt.savefig(outFig, dpi=300, bbox_inches="tight")
        print(f"figure -> {outFig}")
    plt.show()
# endregion

In [ ]:
# region -> release GPU memory
# ---- release GPU memory --------------------------------------------------------------
# Drop references to the big result arrays, clear JAX's compiled caches, and (with
# XLA_PYTHON_CLIENT_ALLOCATOR=platform from the device cell) hand the VRAM back to the
# driver -- no kernel restart needed. A kernel restart is still the guaranteed full reset.
import gc
for _v in ("results", "rawTraces", "obsMatrix", "paramMatrix", "sampled_params",
           "runWall", "states", "structures", "modelObjects"):
    globals().pop(_v, None)
jax.clear_caches()
gc.collect()
try:
    used = sum((d.memory_stats() or {}).get("bytes_in_use", 0) for d in jax.devices())
    print(f"GPU bytes in use after cleanup: {used/1e6:.0f} MB (restart kernel for a full reset)")
except Exception as e:
    print("memory_stats() unavailable on this device:", e)
# endregion